In [2]:
import zarr
import os
import napari

In [23]:
zarr_path = "Y:\\jennifer\\cryolite\\John\\102325_nc281-spiAmSG_cryolite_20Xwi_timelapse_stitched\\102325_nc281-spiAmSG_cryolite_20Xwi_timelapse_stitched_n2v.zarr"

img = zarr.open(zarr_path, mode="r")
# img = img['0']['0']
print(img.shape, img.dtype)

(20, 2, 20, 3229, 3225) >u2


In [24]:
axes = img.attrs.get("axes")
scale = [axis["scale"] for axis in axes if "scale" in axis]
print(axes)

[{'name': 'time', 'scale': 1.0, 'type': 'time', 'unit': 'second'}, {'name': 'channel', 'scale': 1.0, 'type': 'channel'}, {'name': 'z', 'scale': 4.0000027826086955, 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'scale': 0.65, 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'scale': 0.6500001637209303, 'type': 'space', 'unit': 'micrometer'}]


In [25]:
from skimage.transform import rescale

# Downsample each frame separately to handle 5D data
import numpy as np
downsampled_img = np.zeros((img.shape[0], img.shape[1], img.shape[2]//4, img.shape[3]//4, img.shape[4]//4), dtype=img.dtype)

for t in range(img.shape[0]):
    for c in range(img.shape[1]):
        downsampled_img[t, c] = rescale(img[t, c], scale=0.25, order=3, preserve_range=True, anti_aliasing=True).astype(img.dtype)

# Update scale
for axis in axes[2:5]:
    if "scale" in axis:
        axis["scale"] *= 4

In [44]:
from skimage.measure import block_reduce

# Sum pixels in 4x4x4 blocks for spatial dimensions
# Keep time and channel dimensions unchanged
binned_img = block_reduce(img, block_size=(1, 1, 3, 4, 4), func=np.sum)

# Update scale
axes[2]["scale"] *= 3
for axis in axes[3:5]:
    if "scale" in axis:
        axis["scale"] *= 4
print(binned_img.shape, binned_img.dtype)
print(axes)

(20, 2, 7, 808, 807) uint64
[{'name': 'time', 'scale': 1.0, 'type': 'time', 'unit': 'second'}, {'name': 'channel', 'scale': 1.0, 'type': 'channel'}, {'name': 'z', 'scale': 48.00003339130434, 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'scale': 10.4, 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'scale': 10.400002619534884, 'type': 'space', 'unit': 'micrometer'}]


In [45]:
rocks_img = binned_img[:, 0, :, 0:512, 0:512]  # Select the rock channel
# add a new axis for channel dimension
rocks_img = rocks_img[:, None, ...]
rocks_img.shape

(20, 1, 7, 512, 512)

In [46]:
# invert rock channel
rocks_img_invert = 55535 - rocks_img

In [47]:
viewer = napari.Viewer()
viewer.add_image(rocks_img_invert, name='rocks', colormap='gray', blending='additive', scale=scale)

<Image layer 'rocks' at 0x2d71e3c5c90>

In [37]:
rocks_zarr_path = "Y:\\jennifer\\mhat\\data\\nc281-spiAmSG\\03_rocks_rescale.zarr"
os.makedirs(rocks_zarr_path, exist_ok=True)
rocks_zarr = zarr.open(rocks_zarr_path, mode="a", shape=(20, 1, 5, 512, 512), chunks=(1, 1, 5, 512, 512))
rocks_zarr.attrs["axes"] = axes

print(rocks_zarr.shape)

(20, 1, 5, 512, 512)


In [40]:
rocks_zarr[:] = rocks_img_invert

In [48]:
cells_img = binned_img[:, 1, :, 0:512, 0:512]  # Select the first channel
# add a new axis for channel dimension
cells_img = cells_img[:, None, ...]
cells_img.shape

(20, 1, 7, 512, 512)

In [50]:
viewer = napari.Viewer()
viewer.add_image(cells_img, name='cells', colormap='gray', blending='additive', scale=scale)

<Image layer 'cells' at 0x2d72b7195d0>

In [55]:
cells_zarr_path = "Y:\\jennifer\\mhat\\data\\nc281-spiAmSG\\01_cells_binned.zarr"
os.makedirs(cells_zarr_path, exist_ok=True)
cells_zarr = zarr.open(cells_zarr_path, mode="a", shape=(20, 1, 7, 512, 512), chunks=(1, 1, 7, 512, 512))
cells_zarr.attrs["axes"] = axes

print(cells_zarr.shape)

(20, 1, 7, 512, 512)


In [56]:
cells_zarr[:] = cells_img

In [ ]:
import zarr
scaled_zarr_path = "Y:\\jennifer\\mhat\\experiments\\tracking\\nc281-spiAmSG\\02_cells\\2025-11-05_11-16-51\\pred_seg.zarr"

scaled_zarr = zarr.open(scaled_zarr_path, mode="r")

axes = scaled_zarr.attrs.get("axes")
print(axes)

scale = [axis["scale"] for axis in axes if "scale" in axis]
print(scale)

[{'name': 'time', 'scale': 1.0, 'type': 'time', 'unit': 'second'}, {'name': 'z', 'scale': 4.0, 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'scale': 0.65, 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'scale': 0.65, 'type': 'space', 'unit': 'micrometer'}]
[1.0, 4.0, 0.65, 0.65]


In [21]:
import napari

viewer = napari.Viewer()
viewer.add_labels(scaled_zarr, name='segmentation', scale=scale)

<Labels layer 'segmentation' at 0x289c61b4e10>